## Setups and Imports

In [ ]:
# Standard Library Imports
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Add src directory to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Fuzzy string matching
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Project utilities
from utils.helpers import load_orders, set_style, save_figure

set_style()

print("All imports successful!")
print(f"Project root: {project_root}")

## Load Data

In [ ]:
# Load the processed file from Notebook 2
# This file has IS_ANOMALY, FULFILMENT_DAYS, IS_ZERO_PRICE,
# IS_PRICE_OUTLIER and IS_CURRENCY_SUSPECT already added

processed_path = project_root / 'data' / 'processed'
input_file = processed_path / 'orders_with_price_flags.csv'

df_raw = pd.read_csv(input_file, parse_dates=['PURCHASE_TS', 'SHIP_TS'])
df = df_raw.copy()


print(f'Rows:    {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'\nColumns: {list(df.columns)}')

In [ ]:
# PURCHASE_TS should be datetime, but we have some weird values in there that are causing parsing issues. 
# Let's coerce them to datetime and see what we get.
# Simply coerce — NaN stays NaN, everything else becomes datetime

df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')

## Fix Product Names

In [ ]:
# Get all unique product names and their counts

product_counts = df['PRODUCT_NAME'].value_counts()
for name, count in product_counts.items():
    print(f"{count:>6,} - {name}")
    
print(f"\nTotal unique product names: {df['PRODUCT_NAME'].nunique()}")
print(f'Expected unique products: 9(based on domain knowledge)')

In [ ]:
# Fuzzy matching to find similar product names

product_names = df['PRODUCT_NAME'].unique().tolist()

print("\nFuzzy matching results:")
print('Scores above 80 indicate strong similarity and likely duplicates:')

matches = []
for i, name in enumerate(product_names):
    for other_name in product_names[i+1:]:
        score = fuzz.token_sort_ratio(name, other_name)
        matches.append({
            'Name1': name,
            'Name2': other_name,
            'Score': score
        })
        
matches_df = pd.DataFrame(matches).sort_values(by='Score', ascending=False)

# Matches with score above 60, which may indicate potential duplicates or similar products

similar_products = matches_df[matches_df['Score'] > 60]
print(similar_products.to_string(index=False))

In [ ]:
# Confirm duplicate pair in detail
# Determine when each product name was used in the orders 
# and whether one is an old name and the other a new name

duplicate_names = ['27in 4K gaming monitor', '27inches 4k gaming monitor']

for name in duplicate_names:
    subset = df[df['PRODUCT_NAME'] == name]
    print(f"\nOrders with product name: '{name}'")
    print(f' Order count:      {len(subset):,}')
    print(f' Date range:       {subset["PURCHASE_TS"].dropna().min()} to {subset["PURCHASE_TS"].dropna().max()}')
    print(f' Unique order IDs: {subset["PRODUCT_ID"].nunique():,}')
    print(f' Product IDs used: {sorted(subset["PRODUCT_ID"].unique().tolist())}')
    print(f' Median price:     ${subset["USD_PRICE"].median():,.2f}')
    print(f' Price range:      ${subset["USD_PRICE"].min():,.2f} to ${subset["USD_PRICE"].max():,.2f}')

## Product ID Problem

In [ ]:
# How many unique product IDs does each product name have?

print('Product ID counts per product name:')
print('A well governed dataset should have a 1:1 relationship between product name and product ID\n')

id_check = df.groupby('PRODUCT_NAME').agg(
    order_count=('ORDER_ID', 'count'),
    unique_ids=('PRODUCT_ID', 'nunique'),
    id_list=('PRODUCT_ID', lambda x: sorted(x.unique().tolist()))
).sort_values(by='unique_ids', ascending=False)

print(id_check[['order_count', 'unique_ids']].to_string())
print(f'\nTotal unique product names: {df["PRODUCT_NAME"].nunique()}')

In [ ]:
# For each product, show which product IDs it uses and how many
# orders each ID has — this tells us which ID is the primary one
print('Product ID usage breakdown')
print('(Most frequent ID per product = likely the canonical one)\n')

for product in df['PRODUCT_NAME'].unique():
    subset = df[df['PRODUCT_NAME'] == product]
    id_counts = subset['PRODUCT_ID'].value_counts()
    print(f'{product}  ({len(subset):,} orders)')
    for pid, count in id_counts.items():
        pct = count / len(subset) * 100
        bar = '█' * int(pct / 2)
        print(f'  {pid}  {count:>5,} orders ({pct:>5.1f}%)  {bar}')
    print()

## Mapping Tables for product name and product IDs

In [ ]:
# Name mapping table
# Maps every known product name to its canonical version
# '27in 4K gaming monitor' is chosen as canonical because:
#   - It has 4,662 orders vs 61 for the legacy name
#   - It uses the standard abbreviated format consistent with other product names

name_mapping = {
    '27in 4K gaming monitor':          '27in 4K gaming monitor',      # canonical — no change
    '27inches 4k gaming monitor':      '27in 4K gaming monitor',      # legacy → canonical
    'Nintendo Switch':                 'Nintendo Switch',
    'Sony PlayStation 5 Bundle':       'Sony PlayStation 5 Bundle',
    'JBL Quantum 100 Gaming Headset':  'JBL Quantum 100 Gaming Headset',
    'Dell Gaming Mouse':               'Dell Gaming Mouse',
    'Acer Nitro V Gaming Laptop':      'Acer Nitro V Gaming Laptop',
    'Lenovo IdeaPad Gaming 3':         'Lenovo IdeaPad Gaming 3',
    'Razer Pro Gaming Headset':        'Razer Pro Gaming Headset',
}

# Convert to DataFrame for inspection and export
name_mapping_df = pd.DataFrame([
    {'original_name': k, 'canonical_name': v, 'is_duplicate': k != v}
    for k, v in name_mapping.items()
])

print('Product name mapping table')
print(name_mapping_df.to_string(index=False))

In [ ]:
# Product ID mapping table
# For each product, we identify the canonical ID as the one with
# the most orders — the most frequently used ID is the current standard

print('Canonical ID = most frequently used ID per product\n')

# First apply the name fix so the duplicate monitor is treated as one product
df['PRODUCT_NAME_CLEAN'] = df['PRODUCT_NAME'].map(name_mapping)

id_mapping_rows = []
for product in sorted(df['PRODUCT_NAME_CLEAN'].unique()):
    subset = df[df['PRODUCT_NAME_CLEAN'] == product]
    id_counts = subset['PRODUCT_ID'].value_counts()

    # Most frequent ID is the canonical one
    canonical_id = id_counts.index[0]

    for pid in id_counts.index:
        id_mapping_rows.append({
            'product_name_clean': product,
            'original_product_id': pid,
            'canonical_product_id': canonical_id,
            'order_count': id_counts[pid],
            'is_canonical': pid == canonical_id
        })

id_mapping_df = pd.DataFrame(id_mapping_rows)

print(f'Total product IDs mapped: {len(id_mapping_df)}')
print(f'Unique canonical IDs:     {id_mapping_df["canonical_product_id"].nunique()}')
print()
print('Summary: canonical ID per product')
canonical_summary = id_mapping_df[id_mapping_df['is_canonical']][['product_name_clean', 'canonical_product_id', 'order_count']]
print(canonical_summary.to_string(index=False))

## Applying the mappings

In [ ]:
# Build a lookup: original_product_id → canonical_product_id
id_lookup = dict(zip(
    id_mapping_df['original_product_id'],
    id_mapping_df['canonical_product_id']
))

# Apply both mappings to the working dataframe
# PRODUCT_NAME_CLEAN was already created above
# Now add PRODUCT_ID_CLEAN
df['PRODUCT_ID_CLEAN'] = df['PRODUCT_ID'].map(id_lookup)

# Verify the mappings worked
print('Verification: before vs after')
print()
print('Product names — before:')
print(f'  Unique names: {df["PRODUCT_NAME"].nunique()}')
print(f'  Names: {sorted(df["PRODUCT_NAME"].unique().tolist())}')
print()
print('Product names — after:')
print(f'  Unique names: {df["PRODUCT_NAME_CLEAN"].nunique()}')
print(f'  Names: {sorted(df["PRODUCT_NAME_CLEAN"].unique().tolist())}')
print()
print('Product IDs — before:')
print(f'  Unique IDs: {df["PRODUCT_ID"].nunique()}')
print()
print('Product IDs — after:')
print(f'  Unique IDs: {df["PRODUCT_ID_CLEAN"].nunique()}')

In [ ]:
# Confirm order counts are preserved after the name merge
# The two monitor variants should now combine into one row
print('Order counts per product after cleaning')
clean_counts = df['PRODUCT_NAME_CLEAN'].value_counts()
raw_counts   = df['PRODUCT_NAME'].value_counts()

print(f'Products before: {raw_counts.shape[0]}')
print(f'Products after:  {clean_counts.shape[0]}')
print()
for name, count in clean_counts.items():
    print(f'  {count:>6,} orders  →  {name}')
print()
print(f'Total orders: {clean_counts.sum():,} (should match original {len(df):,})')